In [157]:
import sys
import os
from pyflamegpu import *
import pyflamegpu.codegen
import sys
# 切换到你的项目根目录
project_root = r"D:\programming\python_script\socio-physical system for shelter"
os.chdir(project_root)

# 添加 data/output 目录到 Python 路径
sys.path.append('data/output')

In [158]:
from pyflamegpu import *
import pyflamegpu.codegen
import sys
import math
import random

model = pyflamegpu.ModelDescription("F_MAP_tutorial")


# Define an host function called directed_graph_hostfn
class directed_graph_hostfn(pyflamegpu.HostFunction):
  def run(self,FLAMEGPU):
    # Fetch a handle to the directed graph
    fgraph = FLAMEGPU.environment.getDirectedGraph("fgraph")
    # Import a different graph
    fgraph.importGraph("data/env_data/expanded_visibility_graph_renumbered.json");








In [159]:
# Define an host function called write_env_hostfn
class write_env_hostfn(pyflamegpu.HostFunction):
  
  def __init__(self):
    super().__init__()  
  
  def run(self,FLAMEGPU):

      # Retrieve the environment macro property bar of type int array[5][5]

      # Update some of the values
      # foo = 12.0; is not allowed
      FLAMEGPU.environment.importMacroProperty("map", "data/env_data/attraction_matrix_familiar_points.json");

## message

In [160]:

# Define a message of type MessageSpatial2D named location
# MessageSpatial2D: Each agent outputs a message at a specific location in 2D space
# agents only read messages located close to a particular search origin（搜素的中心点）.
# 可以获取一定距离内的消息
message = model.newMessageSpatial3D("location")
# Configure the message list

message.setMin(0, 0,0)
message.setMax(500, 500,100)
message.setRadius(200)
# Add extra variables to the message
# X Y (Z) are implicit for spatial messages
message.newVariableID("id")



In [161]:
stairwell_message = model.newMessageSpatial3D("stairwell_location")
# Configure the message list

stairwell_message.setMin(0, 0,0)
stairwell_message.setMax(500, 500,100)
stairwell_message.setRadius(200)
# Add extra variables to the message
# X Y (Z) are implicit for spatial messages
stairwell_message.newVariableID("id")
stairwell_message.newVariableInt("building_id")
stairwell_message.newVariableFloat("outstop_x")
stairwell_message.newVariableFloat("outstop_y")
stairwell_message.newVariableInt("graph_id")

In [162]:
shelter_message = model.newMessageSpatial3D("shelter_location")
shelter_message.setMin(0, 0,0)
shelter_message.setMax(1000, 1000,200)
shelter_message.setRadius(1000)
shelter_message.newVariableID("id")
shelter_message.newVariableInt("shelter_id")
shelter_message.newVariableInt("graph_id")

In [163]:
# Assign the agent some variables (ID is implicit to agents, so we don't define it ourselves)
student_agent = model.newAgent("student_agent")
student_agent.newVariableFloat("x")
student_agent.newVariableFloat("y")
student_agent.newVariableInt("building_id")
student_agent.newVariableInt("point_id")
student_agent.newVariableFloat("z")
student_agent.newVariableFloat("drift", 0)
student_agent.newVariableInt("target_stairwell_id", -1)
student_agent.newVariableFloat("target_stairwell_x")
student_agent.newVariableFloat("target_stairwell_y")
student_agent.newVariableInt("target_shelter_id", -1)
student_agent.newVariableInt("is_set_shelter", -1)
student_agent.newVariableInt("evacuate_status", 1)
student_agent.newVariableInt("zigzag_dir", 1)
student_agent.newVariableFloat("target_shelter_x")
student_agent.newVariableFloat("target_shelter_y")
# stairwell-connected out stop point:
student_agent.newVariableFloat("outstop_x")
student_agent.newVariableFloat("outstop_y")

student_agent.newVariableInt("start_id")
student_agent.newVariableInt("end_id")
# for road planning





student_agent.newVariableFloat("bar_0_0")
student_agent.newVariableInt("start_vertex_id",131)
student_agent.newVariableInt("end_vertex_id",175)
student_agent.newVariableInt("path_length")
student_agent.newVariableArrayInt("shortest_path", 20)  # 存储最短路径的顶点ID数组，16个顶点的图最长路径不超过20





student_agent.newVariableArrayFloat("shortest_value",176, [-1.0] * 176)

# Dijkstra算法需要的变量
student_agent.newVariableArrayFloat("distances", 176, [999999.0] * 176)  # 距离数组，初始化为无穷大
student_agent.newVariableArrayInt("visited", 176, [0] * 176)  # 访问标记数组
student_agent.newVariableArrayInt("previous", 176, [-1] * 176)  # 前驱节点数组
student_agent.newVariableInt("vertex_count")  # 顶点数量
student_agent.newVariableInt("edge_count")  # 边数量
student_agent.newVariableInt("is_set_shortest_path", 0)

#for move
student_agent.newVariableInt("path_point", 1)


# state shift
student_agent.newState("in_shelter")


In [164]:
stairwell_agent = model.newAgent("stairwell_agent")
stairwell_agent.newVariableFloat("x")
stairwell_agent.newVariableFloat("y")
stairwell_agent.newVariableInt("stairwell_id")
stairwell_agent.newVariableInt("building_id")
stairwell_agent.newVariableFloat("outstop_x")
stairwell_agent.newVariableFloat("outstop_y")
stairwell_agent.newVariableInt("graph_id")
stairwell_agent.newVariableFloat("z")


In [165]:
shelter_agent = model.newAgent("shelter_agent")
shelter_agent.newVariableFloat("x")
shelter_agent.newVariableFloat("y")
shelter_agent.newVariableInt("shelter_id")
shelter_agent.newVariableFloat("z")
shelter_agent.newVariableInt("graph_id")


In [166]:


# Fetch the model's environment
env = model.Environment()
# Declare a new directed graph named 'fgraph'
fgraph = env.newDirectedGraph("fgraph")
# Attach an float[2] property 'bar' to vertices
fgraph.newVertexPropertyArrayFloat("bar", 2)
# Attach an int property 'foo' to edges
fgraph.newEdgePropertyFloat("foo")
env.newMacroPropertyFloat("map", 700, 700)

# 首先我要import一个graph



In [167]:

@pyflamegpu.agent_function
def output_message(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    message_out.setVariableUInt("id", pyflamegpu.getID())
    message_out.setLocation(
        pyflamegpu.getVariableFloat("x"),
        pyflamegpu.getVariableFloat("y"),
        pyflamegpu.getVariableFloat("z")
        )
    return pyflamegpu.ALIVE


In [168]:
@pyflamegpu.agent_function
def stairwell_output_message(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    message_out.setVariableUInt("id", pyflamegpu.getID())
    message_out.setVariableInt("building_id", pyflamegpu.getVariableInt("building_id"))
    message_out.setVariableFloat("outstop_x", pyflamegpu.getVariableFloat("outstop_x"))
    message_out.setVariableFloat("outstop_y", pyflamegpu.getVariableFloat("outstop_y"))
    message_out.setVariableInt("graph_id", pyflamegpu.getVariableInt("graph_id"))
    message_out.setLocation(
        pyflamegpu.getVariableFloat("x"),
        pyflamegpu.getVariableFloat("y"),
        pyflamegpu.getVariableFloat("z")
        )
    return pyflamegpu.ALIVE

In [169]:
@pyflamegpu.agent_function
def shelter_output_message(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    message_out.setVariableUInt("id", pyflamegpu.getID())
    message_out.setVariableInt("shelter_id", pyflamegpu.getVariableInt("shelter_id"))
    message_out.setLocation(
        pyflamegpu.getVariableFloat("x"),
        pyflamegpu.getVariableFloat("y"),
        pyflamegpu.getVariableFloat("z")
        )
    message_out.setVariableInt("graph_id", pyflamegpu.getVariableInt("graph_id"))
    return pyflamegpu.ALIVE

In [170]:
@pyflamegpu.agent_function
def set_target_stairwell(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageNone):
    # Get this agent's x, y, z variables
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    z = pyflamegpu.getVariableFloat("z")
    target_stairwell_id = pyflamegpu.getVariableInt("target_stairwell_id")

    min_dist = 100000
    if target_stairwell_id == -1:
        for message in message_in(x,y,z):
            # Process the message's variables e.g.
            if message.getVariableInt("building_id") == pyflamegpu.getVariableInt("building_id"):
                #找到距离最近的楼梯
                # 找到距离最近的楼梯

                stairwell_x = message.getVariableFloat("x")
                stairwell_y = message.getVariableFloat("y")

                # 计算欧氏距离
                dx = stairwell_x - x
                dy = stairwell_y - y

                dist = math.sqrtf(dx*dx + dy*dy)
                if dist < min_dist:
                    min_dist = dist
                    nearest_stairwell_id = message.getVariableInt("id")
                    # 记录最近楼梯的坐标
                    pyflamegpu.setVariableInt("target_stairwell_id", nearest_stairwell_id)
                    pyflamegpu.setVariableFloat("target_stairwell_x", stairwell_x)
                    pyflamegpu.setVariableFloat("target_stairwell_y", stairwell_y)
                    pyflamegpu.setVariableInt("start_id", message.getVariableInt("graph_id"))
                    pyflamegpu.setVariableFloat("outstop_x", message.getVariableFloat("outstop_x"))
                    pyflamegpu.setVariableFloat("outstop_y", message.getVariableFloat("outstop_y")) 

            #设置目标楼梯
    return pyflamegpu.ALIVE

In [171]:
@pyflamegpu.agent_function
def set_target_shelter_first(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageNone):
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    z = pyflamegpu.getVariableFloat("z")
    
    max_value = 0

    if pyflamegpu.getVariableInt("is_set_shelter") == -1:
        for message in message_in(x,y,z):
            shelter_x = message.getVariableFloat("x")
            shelter_y = message.getVariableFloat("y")

            dx = shelter_x - x  
            dy = shelter_y - y
            dist = math.sqrtf(dx*dx + dy*dy)
            dist_value = math.sqrtf(100/dist)
            m=int(shelter_x)
            n=int(shelter_y)
            map = pyflamegpu.environment.getMacroPropertyFloat("map", 700,700)
            value_familiarity = math.sqrtf(map[m][n])
            value = value_familiarity + dist_value 
            if value > max_value:
                max_value = value
                pyflamegpu.setVariableInt("target_shelter_id", message.getVariableInt("shelter_id"))
                pyflamegpu.setVariableFloat("target_shelter_x", shelter_x)
                pyflamegpu.setVariableFloat("target_shelter_y", shelter_y)
                pyflamegpu.setVariableInt("is_set_shelter", 1)
                pyflamegpu.setVariableInt("end_id", message.getVariableInt("graph_id"))   

    
    return pyflamegpu.ALIVE

                


In [172]:

### 好消息，根据我的test_set_get_in_onefunc.py，我可以实现动态的数据存储啦！！！
@pyflamegpu.agent_function
def ShortestPathFn(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageNone):
    """
    使用Dijkstra算法实现最短路径规划
    """
    fgraph = pyflamegpu.environment.getDirectedGraph("fgraph")

    # 定义常量
    INF = 999999.0
    vertex_count = 176  # 假设图中有16个顶点

    # 获取起点和终点ID（可以从agent变量中获取，或者设置为固定值）
    start_vertex_id = pyflamegpu.getVariableInt("end_id")
    end_vertex_id = pyflamegpu.getVariableInt("start_id")

    ###
    # 起点是哪个，我们就把它的list对应的i，设置为0。
    ###
    pyflamegpu.setVariableFloatArray176("shortest_value", start_vertex_id, 0)

    #然后遍历所有顶点（1-16），除了我们的起点id外，如果它和起点id有连线，我们就把它的list对应的i，设置为起点到它的距离。

    #遍历起点的id，线的值就设置为shortest_value的起点对应的i的值+边的值。
    # 先初始化所有顶点的shortest_value为INF
    for i in range(176):
        if i != start_vertex_id:
            pyflamegpu.setVariableFloatArray176("shortest_value", i, INF)

    # 遍历起点连接的边，设置对应的shortest_value
    for edge in fgraph.outEdges(start_vertex_id):
        # 获取边的目的顶点索引
        dest_vertex_index = edge.getEdgeDestination()
        # 获取边的foo属性
        foo = edge.getPropertyInt("foo")

        # 将foo值存储到对应顶点ID的位置
        pyflamegpu.setVariableFloatArray176("shortest_value", dest_vertex_index, foo)
            


    # 获取起点和终点的索引
    start_index = fgraph.getVertexIndex(start_vertex_id)
    end_index = fgraph.getVertexIndex(end_vertex_id)

    # 初始化起点距离为0
    pyflamegpu.setVariableFloatArray176("distances", start_index, 0.0)
    # 初始化起点前驱为-1
    pyflamegpu.setVariableIntArray176("previous", start_index, -1)
    # 初始化其他顶点的距离为无穷大，前驱为-1
    for i in range(vertex_count):
        if i != start_index:
            pyflamegpu.setVariableFloatArray176("distances", i, INF)
            pyflamegpu.setVariableIntArray176("previous", i, -1)
        pyflamegpu.setVariableIntArray176("visited", i, 0)

    # Dijkstra算法主循环
    for _ in range(vertex_count):
        # 找到当前未访问顶点中距离最小的顶点
        min_distance = INF      
        current_index = -1

        for i in range(vertex_count):
            visited_i = pyflamegpu.getVariableIntArray176("visited", i)
            distance_i = pyflamegpu.getVariableFloatArray176("distances", i) 
            if visited_i == 0 and distance_i < min_distance:
                min_distance = distance_i
                current_index = i

        if current_index == -1 or min_distance == INF:
            break

        # 标记当前顶点为已访问
        pyflamegpu.setVariableIntArray176("visited", current_index, 1)   

        # 如果到达终点，可以提前结束
        if current_index == end_index:
            break


        # 遍历当前顶点的所有出边
        for edge in fgraph.outEdges(current_index):

            dest_index = edge.getEdgeDestination()

            visited_dest = pyflamegpu.getVariableIntArray176("visited", dest_index)
            if visited_dest == 0:
                # 获取边的权重
                edge_weight = edge.getPropertyFloat("foo")
                current_distance = pyflamegpu.getVariableFloatArray176("distances", current_index)
                new_distance = current_distance + edge_weight

                dest_distance = pyflamegpu.getVariableFloatArray176("distances", dest_index)
                if new_distance < dest_distance:
                    pyflamegpu.setVariableFloatArray176("distances", dest_index, new_distance)
                    pyflamegpu.setVariableIntArray176("previous", dest_index, current_index)

    # 重建最短路径 - 使用pyflamegpu方法完全避免循环和索引操作

    # 初始化路径数组为-1
    for i in range(20):
        pyflamegpu.setVariableIntArray20("shortest_path", i, -1)

    # 从终点开始重建路径
    current = end_index
    path_index = 0
    path_length = 0

    # 逆向追溯直到起点或前驱为-1
    while current != -1 and path_index < 20:
        vertex_id = fgraph.getVertexID(current)
        pyflamegpu.setVariableIntArray20("shortest_path", path_index, vertex_id)
        path_index += 1
        path_length += 1
        
        # 如果到达起点，停止追溯
        if current == start_index:  # 假设start_index是起点的索引
            break
        
        # 获取前驱节点
        current = pyflamegpu.getVariableIntArray176("previous", current)

    # 设置路径长度
    pyflamegpu.setVariableInt("path_length", path_length)

    # 验证路径是否确实从终点连接到起点
    first_vertex = pyflamegpu.getVariableIntArray20("shortest_path", 0)
    last_vertex = pyflamegpu.getVariableIntArray20("shortest_path", path_length - 1)

    if first_vertex != end_vertex_id or last_vertex != start_vertex_id:
        # 路径不完整，可能需要特殊处理
        pyflamegpu.setVariableInt("path_length", 0)  # 或者标记为无效路径

    pyflamegpu.setVariableInt("is_set_shortest_path", 1)
    return pyflamegpu.ALIVE 



In [173]:
@pyflamegpu.agent_function_condition
def is_set_shortest_path_is_1() -> bool:
    return pyflamegpu.getVariableInt("is_set_shortest_path") == 0

In [174]:
import math

In [175]:
@pyflamegpu.agent_function
def move_to_stairwell(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    target_stairwell_x = pyflamegpu.getVariableFloat("target_stairwell_x")
    target_stairwell_y = pyflamegpu.getVariableFloat("target_stairwell_y")
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    
    delta_x = target_stairwell_x - x
    delta_y = target_stairwell_y - y

    distance = math.sqrtf(delta_x * delta_x + delta_y * delta_y)

    next_x=0.0
    next_y=0.0
    if distance > 1.0:
        step_x = delta_x / distance 
        step_y = delta_y / distance
        next_x = x + step_x
        next_y = y + step_y
    else:
        next_x =  target_stairwell_x
        next_y =  target_stairwell_y
        pyflamegpu.setVariableInt("evacuate_status", 2)

    pyflamegpu.setVariableFloat("x", next_x)
    pyflamegpu.setVariableFloat("y", next_y)

    
    message_out.setLocation(next_x, next_y, pyflamegpu.getVariableFloat("z"))


    return pyflamegpu.ALIVE

In [176]:
#conditional function set
@pyflamegpu.agent_function_condition
def eva_state_is_1() -> bool :
    return pyflamegpu.getVariableInt("evacuate_status") == 1

In [177]:
# z字形下楼的算法
# 假设每层楼高3，行人和stairwell的x、y初始一致
# 通过y,z索引，z字形移动，每次移动一小步，遇到转折点y反向

@pyflamegpu.agent_function
def down_stairwell(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    """
    行人在stairwell点z字形下降，每层楼高3
    通过y,z索引，z字形移动
    """
    # 获取当前位置
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    z = pyflamegpu.getVariableFloat("z")
    # 获取目标stairwell的x,y
    stairwell_x = pyflamegpu.getVariableFloat("target_stairwell_x")
    stairwell_y = pyflamegpu.getVariableFloat("target_stairwell_y")
    # 获取当前z字形方向（1为y正向，-1为y负向）
    if not pyflamegpu.getVariableInt("zigzag_dir"):
        pyflamegpu.setVariableInt("zigzag_dir", 1)

    dire = pyflamegpu.getVariableInt("zigzag_dir")
    # 每步y方向移动距离
    step_y = 0.5
    # 每步z方向下降距离
    step_z = 0.2
    # z字形的y范围（比如以stairwell_y为中心，上下各1.5）
    y_min = stairwell_y - 1.5
    y_max = stairwell_y + 1.5
    # 计算下一步y
    next_y = y + dire * step_y
    # 判断是否到达边界，若到达则反向
    if next_y > y_max:
        next_y = y_max
        dire = -1
    elif next_y < y_min:
        next_y = y_min
        dire = 1
    # 计算下一步z
    next_z = z - step_z
    # 判断是否到达下一层（z是否小于目标z）
    # 假设目标z为0
    if next_z < 0:
        next_z = 0
        pyflamegpu.setVariableInt("evacuate_status", 3)
    # 更新变量
    pyflamegpu.setVariableFloat("y", next_y)
    pyflamegpu.setVariableFloat("z", next_z)
    pyflamegpu.setVariableInt("zigzag_dir", dire)
    # x保持不变
    pyflamegpu.setVariableFloat("x", stairwell_x)
    # 输出当前位置
    message_out.setLocation(stairwell_x, next_y, next_z)
    return pyflamegpu.ALIVE

    

In [178]:
#conditional function set
@pyflamegpu.agent_function_condition
def eva_state_is_2() -> bool :
    return pyflamegpu.getVariableInt("evacuate_status") == 2

In [179]:
@pyflamegpu.agent_function
def move_to_stop(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageNone):
    outstop_x = pyflamegpu.getVariableFloat("outstop_x")
    outstop_y = pyflamegpu.getVariableFloat("outstop_y")
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")
    
    delta_x = outstop_x - x
    delta_y = outstop_y - y

    distance = math.sqrtf(delta_x * delta_x + delta_y * delta_y)

    next_x=0.0
    next_y=0.0
    if distance > 1.0:
        step_x = delta_x / distance 
        step_y = delta_y / distance
        next_x = x + step_x
        next_y = y + step_y
    else:
        next_x =  outstop_x
        next_y =  outstop_y
        pyflamegpu.setVariableInt("evacuate_status", 4)

    pyflamegpu.setVariableFloat("x", next_x)
    pyflamegpu.setVariableFloat("y", next_y)

    return pyflamegpu.ALIVE

In [180]:
#conditional function set
@pyflamegpu.agent_function_condition
def eva_state_is_3() -> bool :
    return pyflamegpu.getVariableInt("evacuate_status") == 3

In [181]:
@pyflamegpu.agent_function
def move_to_shelter(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    
    fgraph = pyflamegpu.environment.getDirectedGraph("fgraph")
    i = pyflamegpu.getVariableInt("path_point")
    m = pyflamegpu.getVariableIntArray20("shortest_path", i)
    n = pyflamegpu.getVariableIntArray20("shortest_path", i+1)

    target_shelter_x = fgraph.getVertexPropertyFloatArray2("bar", m-1, 0)
    target_shelter_y = fgraph.getVertexPropertyFloatArray2("bar", m-1, 1)
    x = pyflamegpu.getVariableFloat("x")
    y = pyflamegpu.getVariableFloat("y")


    delta_x = target_shelter_x - x
    delta_y = target_shelter_y - y

    distance = math.sqrtf(delta_x * delta_x + delta_y * delta_y)

    next_x=0.0
    next_y=0.0
    if distance > 1.0:
        step_x = delta_x / distance 
        step_y = delta_y / distance
        next_x = x + step_x
        next_y = y + step_y
    else:
        next_x =  target_shelter_x
        next_y =  target_shelter_y
        if n == -1:
            pyflamegpu.setVariableInt("evacuate_status", 5)
        else:
            pyflamegpu.setVariableInt("path_point", i+1)

    pyflamegpu.setVariableFloat("x", next_x)
    pyflamegpu.setVariableFloat("y", next_y)

    message_out.setLocation(next_x, next_y, pyflamegpu.getVariableFloat("z"))




    return pyflamegpu.ALIVE

In [182]:
#conditional function set
@pyflamegpu.agent_function_condition
def eva_state_is_4() -> bool :
    return pyflamegpu.getVariableInt("evacuate_status") == 4

In [183]:
@pyflamegpu.agent_function
def in_shelter(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageNone):

    return pyflamegpu.ALIVE


In [184]:
#conditional function set
@pyflamegpu.agent_function_condition
def eva_state_is_5() -> bool :
    return pyflamegpu.getVariableInt("evacuate_status") == 5

In [185]:

output_func_translated = pyflamegpu.codegen.translate(output_message)
stairwell_output_func_translated = pyflamegpu.codegen.translate(stairwell_output_message)
shelter_output_func_translated = pyflamegpu.codegen.translate(shelter_output_message)
set_target_stairwell_func_translated = pyflamegpu.codegen.translate(set_target_stairwell)
set_target_shelter_func_translated = pyflamegpu.codegen.translate(set_target_shelter_first)
#ExampleFn_translated = pyflamegpu.codegen.translate(ExampleFn)
move_to_stairwell_func_translated = pyflamegpu.codegen.translate(move_to_stairwell)


ShortestPathFn_translated = pyflamegpu.codegen.translate(ShortestPathFn)
down_stairwell_func_translated = pyflamegpu.codegen.translate(down_stairwell)
move_to_stop_func_translated = pyflamegpu.codegen.translate(move_to_stop)
in_shelter_func_translated = pyflamegpu.codegen.translate(in_shelter)

eva_state_is_1_func_translated = pyflamegpu.codegen.translate(eva_state_is_1)
eva_state_is_2_func_translated = pyflamegpu.codegen.translate(eva_state_is_2)
eva_state_is_3_func_translated = pyflamegpu.codegen.translate(eva_state_is_3)
eva_state_is_4_func_translated = pyflamegpu.codegen.translate(eva_state_is_4)
eva_state_is_5_func_translated = pyflamegpu.codegen.translate(eva_state_is_5)
is_set_shortest_path_is_1_func_translated = pyflamegpu.codegen.translate(is_set_shortest_path_is_1)




# Setup the two agent functions
output_fn = student_agent.newRTCFunction("output_message", output_func_translated)
output_fn.setMessageOutput("location")

stairwell_output_fn = stairwell_agent.newRTCFunction("stairwell_output_message", stairwell_output_func_translated)
stairwell_output_fn.setMessageOutput("stairwell_location")

shelter_output_fn = shelter_agent.newRTCFunction("shelter_output_message", shelter_output_func_translated)
shelter_output_fn.setMessageOutput("shelter_location")

set_target_stairwell_fn = student_agent.newRTCFunction("set_target_stairwell", set_target_stairwell_func_translated)
set_target_stairwell_fn.setMessageInput("stairwell_location")

set_target_shelter_fn = student_agent.newRTCFunction("set_target_shelter", set_target_shelter_func_translated)
set_target_shelter_fn.setMessageInput("shelter_location")

#ExampleFn_fn = agent.newRTCFunction("ExampleFn",ExampleFn_translated)
ShortestPathFn_fn = student_agent.newRTCFunction("ShortestPathFn",ShortestPathFn_translated)
ShortestPathFn_fn.setRTCFunctionCondition(is_set_shortest_path_is_1_func_translated)

move_to_stairwell_fn = student_agent.newRTCFunction("move_to_stairwell", move_to_stairwell_func_translated)
move_to_stairwell_fn.setMessageOutput("location")
move_to_stairwell_fn.setRTCFunctionCondition(eva_state_is_1_func_translated)

down_stairwell_fn = student_agent.newRTCFunction("down_stairwell", down_stairwell_func_translated)
down_stairwell_fn.setMessageOutput("location")
down_stairwell_fn.setRTCFunctionCondition(eva_state_is_2_func_translated)

move_to_stop_fn = student_agent.newRTCFunction("move_to_stop", move_to_stop_func_translated)
move_to_stop_fn.setRTCFunctionCondition(eva_state_is_3_func_translated)


move_to_shelter_translated = pyflamegpu.codegen.translate(move_to_shelter)
move_to_shelter_fn = student_agent.newRTCFunction("move_to_shelter", move_to_shelter_translated)
move_to_shelter_fn.setMessageOutput("location")
move_to_shelter_fn.setRTCFunctionCondition(eva_state_is_4_func_translated)

in_shelter_fn = student_agent.newRTCFunction("in_shelter", in_shelter_func_translated)
in_shelter_fn.setInitialState("default")
in_shelter_fn.setEndState("in_shelter")
in_shelter_fn.setRTCFunctionCondition(eva_state_is_5_func_translated)



model.addInitFunction(directed_graph_hostfn())
#model.addExecutionRoot(ExampleFn_fn)
model.addExecutionRoot(output_fn)
stairwell_output_fn.dependsOn(output_fn)
shelter_output_fn.dependsOn(output_fn)
set_target_shelter_fn.dependsOn(shelter_output_fn)
set_target_stairwell_fn.dependsOn(stairwell_output_fn)
ShortestPathFn_fn.dependsOn(set_target_shelter_fn)
ShortestPathFn_fn.dependsOn(set_target_stairwell_fn)
move_to_stairwell_fn.dependsOn(ShortestPathFn_fn)

down_stairwell_fn.dependsOn(move_to_stairwell_fn)
move_to_stop_fn.dependsOn(down_stairwell_fn)
move_to_shelter_fn.dependsOn(move_to_stop_fn)
in_shelter_fn.dependsOn(move_to_shelter_fn)


model.generateLayers() 




# Specify the desired StepLoggingConfig
step_log_cfg = pyflamegpu.StepLoggingConfig(model)
# Log every step
step_log_cfg.setFrequency(1)
# Include the mean of the "point" agent population's variable 'drift'


step_log_cfg.agent("student_agent", "in_shelter").logCount()

# Create and init the simulation
cuda_model = pyflamegpu.CUDASimulation(model)



FLAMEGPURuntimeException: (InvalidStateName) D:\a\FLAMEGPU2\FLAMEGPU2\src\flamegpu\model\AgentFunctionDescription.cpp(142): Agent ('student_agent') does not contain state 'default', in AgentFunctionDescription::setInitialState().

In [ ]:
from flamegpu_init_code import initialize_student_agent_population
from shelter_flamegpu_init_code import initialize_shelter_agent_population


import random
shelter_index_vector = random.sample(range(71), 6)  # 假设shelter点索引范围为0~29，请根据实际数量调整
# initialize agent population
initialize_student_agent_population(model, cuda_model)
initialize_shelter_agent_population(model, cuda_model, shelter_index_vector)

初始化 6391 个学生代理个体
❌ 加载人口数据失败: (InvalidAgentState) D:\a\FLAMEGPU2\FLAMEGPU2\src\flamegpu\simulation\detail\CUDAAgent.cu(140): Agent 'student_agent' does not use the default state, so the state must be passed explicitly, in CUDAAgent::setPopulationData()
初始化 28 个楼梯间代理
楼梯间代理种群初始化完成
初始化 6 个shelter代理个体
shelter代理种群初始化完成


In [ ]:



WIDTH=500

# Only run this block if pyflamegpu was built with visualisation support
if pyflamegpu.VISUALISATION:
    # Create visualisation
    m_vis = cuda_model.getVisualisation()
    # Set the initial camera location and speed
    INIT_CAM = WIDTH / 2
    m_vis.setInitialCameraTarget(270, 205, 0)
    m_vis.setInitialCameraLocation(240, 100, 100)
    m_vis.setCameraSpeed(0.01)
    m_vis.setSimulationSpeed(100)
    # Add "point" agents to the visualisation

    
    # Add "student_agent" agents to the visualisation
    student_agt = m_vis.addAgent("student_agent")
    student_agt.setModel(pyflamegpu.ICOSPHERE);
    student_agt.setModelScale(1/1.0);
    # Mark the environment bounds.

    stairwell_agt = m_vis.addAgent("stairwell_agent")
    stairwell_agt.setModel(pyflamegpu.ICOSPHERE);
    stairwell_agt.setModelScale(1/0.5);
    stairwell_agt.setColor(pyflamegpu.RED);

    shelter_agt = m_vis.addAgent("shelter_agent")
    shelter_agt.setModel(pyflamegpu.ICOSPHERE);
    shelter_agt.setModelScale(1/0.2);
    shelter_agt.setColor(pyflamegpu.BLUE);
    
    pen = m_vis.newPolylineSketch(1, 1, 1, 0.2)
    pen.addVertex(275, 637, 0) # 起始点
    pen.addVertex(69, 510, 0)
    pen.addVertex(0, 301, 0)
    pen.addVertex(1, 167, 0)
    pen.addVertex(29, 142, 0)
    pen.addVertex(57, 98, 0)
    pen.addVertex(118, 67, 0)
    pen.addVertex(109, 24, 0)
    pen.addVertex(287, 0, 0)
    pen.addVertex(286, 45, 0)
    pen.addVertex(405, 154, 0)
    pen.addVertex(435, 131, 0)
    pen.addVertex(436, 72, 0)
    pen.addVertex(467, 41, 0)
    pen.addVertex(501, 35, 0)
    pen.addVertex(543, 47, 0)
    pen.addVertex(275, 637, 0) # 闭合点 
    # Open the visualiser window 
    m_vis.activate()

cuda_model.setStepLog(step_log_cfg)
cuda_model.SimulationConfig().steps = 1000
cuda_model.CUDAConfig().device_id = 0
cuda_model.applyConfig()
# Run the simulation


cuda_model.simulate()



if pyflamegpu.VISUALISATION:
    # Keep the visualisation window active after the simulation has completed
    m_vis.join()

In [ ]:
step_log = run_log.getStepLog();

run_log = cuda_model.getRunLog();


step_log = run_log.getStepLog();

for log in step_log:
    # Get the step index
    step_count = log.getStepCount();
    # Get a logged environment property
    agent_count = log.getAgent("student_agent", "in_shelter").getCount()
    # Print data to console
    print("#%u: %u, %f"%(step+count, agent_count, agent_speed_mean))

NameError: name 'run_log' is not defined

In [ ]:
import numpy as np

out_pop = pyflamegpu.AgentVector(model.Agent("student_agent"))
cuda_model.getPopulationData(out_pop)

# 创建结构化数组
dtype = [('start_id', 'i4'),('end_id', 'i4'),('target_shelter_id', 'i4'),('target_stairwell_id', 'i4'),('building_id', 'i4'),('x','f4'),('y','f4'),('z', 'f4'),('shortest_path', 'i4', (20,))]
agent_array = np.array(
    [(agent.getVariableInt("start_id"),agent.getVariableInt("end_id"),agent.getVariableInt("target_shelter_id"), agent.getVariableInt("target_stairwell_id"), agent.getVariableInt("building_id"), agent.getVariableFloat("x"),agent.getVariableFloat("y"),agent.getVariableFloat("z"),agent.getVariableArrayInt("shortest_path")) 
     for agent in out_pop],
    dtype=dtype
)

agent_array

array([(162, 114, 36, 14, 5, 280., 160., 0., [162, 114,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1]),
       (162, 114, 36, 14, 5, 280., 160., 0., [162, 114,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1]),
       (162, 114, 36, 14, 5, 280., 160., 0., [162, 114,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1]),
       ...,
       (175,  86,  8, 27, 0, 140., 200., 0., [175,  86,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1]),
       (175,  86,  8, 27, 0, 140., 200., 0., [175,  86,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1]),
       (175,  86,  8, 27, 0, 140., 200., 0., [175,  86,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1])],
      dtype=[('start_id', '<i4'), ('end_id', '<i4'), ('target_shelter_id', '<i4'), ('target_stairwe